In [ ]:
!pip install -U bitsandbytes transformers accelerate peft datasets evaluate rouge_score sentence-transformers nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.4 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=74ae504b31c696ad813cdfdfb05790fee0bbf076a816dcf5006adfbace951a5e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18

In [ ]:
import os, json, random, re, math, torch
from pathlib import Path
from tqdm import tqdm
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, Trainer, TrainingArguments,
    DataCollatorForSeq2Seq, pipeline
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

ALPACA_TMPL = (
    "### Instruction:\n"
    "Answer as an endodontist using only the given context. "
    "Preserve exact clinical and endodontic keywords from the context whenever possible. "
    "Do not replace technical terms with general language. "
    "Use abbreviations such as NaOCl, EDTA, MTA exactly if they appear in the context.\n\n"
    "### Input:\n"
    "CONTEXT:\n{context}\n\n"
    "Question: {question}\n\n"
    "### Response:\n"
)

def format_for_training(context: str, question: str, answer: str) -> str:
    return ALPACA_TMPL.format(context=context, question=question) + answer

def format_for_inference(context: str, question: str) -> str:
    return ALPACA_TMPL.format(context=context, question=question)

print("Import ve sabitler hazir.")

Import ve sabitler hazir.


In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False
print("Model yuklendi:", model_id)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model yuklendi: mistralai/Mistral-7B-Instruct-v0.2


In [ ]:
chunks_path    = Path("/content/chunks_temiz (1).jsonl")
qa_output_path = Path("/content/raw_qa_dataset.jsonl")

rows = []
with chunks_path.open("r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        if "text" in row and len(row["text"].strip()) > 250:
            rows.append(row)

print(f"Uygun chunk sayisi: {len(rows)}")

Uygun chunk sayisi: 2046


In [ ]:
qa_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

QA_GENERATION_PROMPT = """You are a world-class Endodontist.
Read the given text and produce exactly one question-answer pair.

Rules:
- The question must be answerable only from the given text.
- The answer must be short, clear, and technical.
- The answer must contain at least 2 important endodontic or clinical keywords copied from the text.
- Do not replace technical terms with simpler synonyms.
- Output format must be exactly:

Question: ...
Answer: ...

Text:
{context}
"""

def generate_qa(context: str):
    prompt = QA_GENERATION_PROMPT.format(context=context[:1200])
    outputs = qa_generator(
        prompt,
        max_new_tokens=140,
        do_sample=True,
        temperature=0.4,
        top_p=0.9,
        repetition_penalty=1.1,
        return_full_text=False
    )
    raw = outputs[0]["generated_text"].strip()

    q_match = re.search(r"Question:\s*(.*)", raw)
    a_match = re.search(r"Answer:\s*(.*)", raw, re.DOTALL)

    if not q_match or not a_match:
        return None

    question = q_match.group(1).strip()
    answer   = a_match.group(1).strip()

    if "Question:" in answer:
        answer = answer.split("Question:")[0].strip()

    if len(question) < 12 or len(answer) < 20:
        return None

    return {
        "context":  context[:1200],
        "question": question,
        "answer":   answer
    }

sample_size  = min(800, len(rows))
sampled_rows = random.sample(rows, sample_size)

raw_qa = []
for ch in tqdm(sampled_rows):
    item = generate_qa(ch["text"])
    if item:
        raw_qa.append(item)

print(f"Uretilen ornek sayisi: {len(raw_qa)}")

with qa_output_path.open("w", encoding="utf-8") as f:
    for entry in raw_qa:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")
print(f"Kaydedildi: {qa_output_path}")


  0%|          | 0/800 [00:00<?, ?it/s][transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'temperature', 'repetition_penalty', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=140) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization 

Uretilen ornek sayisi: 763
Kaydedildi: /content/raw_qa_dataset.jsonl


In [ ]:
MAX_LENGTH = 768

def tokenize_with_response_only_loss(example):
    prompt    = ALPACA_TMPL.format(context=example["context"], question=example["question"])
    full_text = prompt + example["answer"]

    full_tokens   = tokenizer(full_text, truncation=True, max_length=MAX_LENGTH, padding="max_length")
    prompt_tokens = tokenizer(prompt,    truncation=True, max_length=MAX_LENGTH, padding=False)

    input_ids      = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]
    labels         = input_ids.copy()

    prompt_len          = len(prompt_tokens["input_ids"])
    labels[:prompt_len] = [-100] * prompt_len
    labels = [lbl if attn == 1 else -100 for lbl, attn in zip(labels, attention_mask)]

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_data     = Dataset.from_list(raw_qa)
tokenized_data = train_data.map(
    tokenize_with_response_only_loss,
    remove_columns=train_data.column_names
)


sample  = tokenized_data[0]
decoded = tokenizer.decode(
    [t for t, l in zip(sample["input_ids"], sample["labels"]) if l != -100],
    skip_special_tokens=True
)
print("Model sadece sunu ogreniyor:")
print(decoded[:300])
print("Tokenization tamamlandi.")

Map:   0%|          | 0/763 [00:00<?, ? examples/s]

Model sadece sunu ogreniyor:
:
Ridge preservation, in periradicular surgery, 460 RinsEndo system, 286 Root apical third of, 199–200 curved, 159f post placement and selection affected by, 877 preparation of, 839–840 extraoral dry time less than 60 minutes, 839 , 839f extraoral dry time more than 60 minutes, 839–840 Root apex, il
Tokenization tamamlandi.


In [ ]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

args = TrainingArguments(
    output_dir="./endodonti_fine_tuned_model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    optim="paged_adamw_8bit"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_data,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True
    )
)

trainer.train()
trainer.model.save_pretrained("./endodonti_fine_tuned_model")
tokenizer.save_pretrained("./endodonti_fine_tuned_model")
print("Egitim tamamlandi.")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.011448
20,1.379889
30,1.296299
40,1.288798
50,1.235699
60,1.052769
70,1.045177
80,1.037333
90,1.055588
100,0.908599


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Egitim tamamlandi.


In [ ]:
import os
dosyalar = os.listdir("./endodonti_fine_tuned_model")
print("Model dosyalari:", dosyalar)
assert "adapter_config.json" in dosyalar, "HATA: adapter_config.json yok!"
print("adapter_config.json mevcut.")

import shutil
from google.colab import drive, files

drive.mount("/content/drive")

shutil.make_archive("endodonti_final_paket", "zip", "./endodonti_fine_tuned_model")
files.download("endodonti_final_paket.zip")

shutil.make_archive(
    "/content/drive/MyDrive/endodonti_final_paket",
    "zip",
    "./endodonti_fine_tuned_model"
)
print("Drive'a kaydedildi.")

Model dosyalari: ['adapter_model.safetensors', 'tokenizer.json', 'chat_template.jinja', 'adapter_config.json', 'checkpoint-144', 'checkpoint-240', 'tokenizer_config.json', 'checkpoint-96', 'checkpoint-192', 'README.md', 'checkpoint-48']
adapter_config.json mevcut.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Drive'a kaydedildi.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def asistan_test_et(soru: str, context: str) -> str:
    model.eval()
    prompt = format_for_inference(context[:1200], soru)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    cevap = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


    for prefix in ["### Response:", "### Input:", "Question:", "Answer:"]:
        if cevap.startswith(prefix):
            cevap = cevap[len(prefix):].strip()
    for kesim in ["\nQuestion:", "\nAnswer:", "\n###"]:
        if kesim in cevap:
            cevap = cevap.split(kesim)[0].strip()

    return cevap

TEST_SORULARI = [
    "What is the main purpose of EDTA in endodontics?",
    "What are the primary objectives of using Sodium Hypochlorite (NaOCl)?",
    "What are the clinical signs of irreversible pulpitis?",
    "How should a clinician manage a separated instrument in the root canal?",
    "Explain the importance of working length determination in endodontics."
]

REFERANSLAR = [
    "EDTA is a chelating agent used to remove the inorganic smear layer and soften dentin walls, improving penetration of irrigants.",
    "NaOCl dissolves organic tissue and has broad-spectrum antimicrobial activity, making it the gold standard irrigant.",
    "Irreversible pulpitis presents as sharp lingering pain to thermal stimuli and may include spontaneous pain.",
    "Management of a separated instrument includes bypass attempts, ultrasonic retrieval, or cleaning to the file level.",
    "Working length determination ensures instrumentation stays within the root canal system, preventing apical damage."
]


metinler = [r["text"] for r in rows]
vect     = TfidfVectorizer(stop_words="english")
tmat     = vect.fit_transform(metinler)

print("Cevaplar uretiliyor...")
tahminler = []
for soru in TEST_SORULARI:
    sv    = vect.transform([soru])
    idx   = cosine_similarity(sv, tmat)[0].argsort()[::-1][0]
    ctx   = rows[idx]["text"][:1000]
    cevap = asistan_test_et(soru, ctx)
    tahminler.append(cevap)
    print(f"  SORU: {soru[:50]}")
    print(f"  CEVAP: {cevap[:150]}\n")

Cevaplar uretiliyor...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  SORU: What is the main purpose of EDTA in endodontics?
  CEVAP: EDTA forms a stablecomplexwith metal iconsfromthecellenvelope,leadingtobacterialdeth,butitcannoteffectivelyremovethesmearlayeralone;itrequiresaproteol

  SORU: What are the primary objectives of using Sodium Hy
  CEVAP: The primary objectives for using Sodiun Hypchlorite in Endodontics include its disinfection and tissue dissolving properties as recommended by Europea

  SORU: What are the clinical signs of irreversible pulpit
  CEVAP: Irreversibly inflamedpulpshowclinicallydefinedsymptomsofpainandmayalsobeasymptomaticapicalperiodontitis.Histologically,theresidualvascularityandnervef

  SORU: How should a clinician manage a separated instrume
  CEVAP: Retrieve the separated file using instruments like the PizyoFlow or ultrasonics, confirm its removal on a radiograph, and then obtain a good cleaning 

  SORU: Explain the importance of working length determina
  CEVAP: Working length determination is crucial for avoiding o

In [ ]:
import pandas as pd
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

def temizle(metin: str) -> str:
    metin = str(metin).lower()
    metin = re.sub(r"[^\w\s]", " ", metin)
    return re.sub(r"\s+", " ", metin).strip()

def hesapla_perplexity(metin: str) -> float:
    enc = tokenizer(metin, return_tensors="pt", truncation=True, max_length=256)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        loss = model(**enc, labels=enc["input_ids"]).loss
    return round(math.exp(loss.item()), 2)

cos_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
scorer    = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smooth    = SmoothingFunction().method1

KLINIK_SOZLUK = {
    0: ["edta", "smear", "layer", "chelat", "inorganic", "calcium", "dentin"],
    1: ["naocl", "hypochlorite", "disinfect", "dissolution", "antimicrobial", "organic", "tissue"],
    2: ["irreversible", "pulpitis", "pain", "thermal", "lingering", "spontaneous"],
    3: ["separated", "instrument", "bypass", "ultrasonic", "retrieval", "file"],
    4: ["working", "length", "cleaning", "shaping", "root", "canal", "apical"]
}

def keyword_capture(cevap, kelimeler):
    t = temizle(cevap)
    bulunan = [k for k in kelimeler if k in t]
    return round(len(bulunan) / len(kelimeler), 3) if kelimeler else 0.0, bulunan

satirlar = []
print("Metrikler hesaplaniyor...")

for i, (ref, pred) in enumerate(zip(REFERANSLAR, tahminler)):
    ref_t    = temizle(ref)
    pred_t   = temizle(pred)
    ref_tok  = ref_t.split()
    pred_tok = pred_t.split()

    bleu    = sentence_bleu([ref_tok], pred_tok, smoothing_function=smooth)
    rouge_l = scorer.score(ref_t, pred_t)["rougeL"].fmeasure
    meteor  = meteor_score([ref_tok], pred_tok)
    ppl     = hesapla_perplexity(pred)
    vecs    = cos_model.encode([ref, pred], normalize_embeddings=True)
    cosine  = float(np.dot(vecs[0], vecs[1]))
    kw_oran, bulunan = keyword_capture(pred, KLINIK_SOZLUK.get(i, []))

    satirlar.append({
        "Soru":             TEST_SORULARI[i][:45] + "...",
        "BLEU":             round(bleu,    3),
        "ROUGE-L":          round(rouge_l, 3),
        "METEOR":           round(meteor,  3),
        "Perplexity":       ppl,
        "Cosine_Similarity":round(cosine,  3),
        "Keyword_Hit_Rate": kw_oran,
        "Bulunan":          ", ".join(bulunan)
    })

df_metrik = pd.DataFrame(satirlar)

print("\n" + "="*55)
print("METRİK ÖZETİ")
print("="*55)
ozet = df_metrik.drop(columns=["Soru", "Bulunan"]).mean(numeric_only=True)
for metrik, deger in ozet.items():
    print(f"  {metrik:<22}: {deger:.3f}")
print("="*55)
print(df_metrik.to_string(index=False))

df_metrik.to_csv("./metrik_sonuclari.csv", index=False)
print("Kaydedildi: metrik_sonuclari.csv")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Metrikler hesaplaniyor...

METRİK ÖZETİ
  BLEU                  : 0.006
  ROUGE-L               : 0.076
  METEOR                : 0.144
  Perplexity            : 26.924
  Cosine_Similarity     : 0.641
  Keyword_Hit_Rate      : 0.519
                                            Soru  BLEU  ROUGE-L  METEOR  Perplexity  Cosine_Similarity  Keyword_Hit_Rate                                 Bulunan
What is the main purpose of EDTA in endodonti... 0.011    0.100   0.050       34.69              0.692             0.571              edta, smear, layer, dentin
What are the primary objectives of using Sodi... 0.005    0.056   0.150       30.59              0.613             0.429                naocl, disinfect, tissue
What are the clinical signs of irreversible p... 0.000    0.048   0.062       23.46              0.533             0.500            irreversible, pulpitis, pain
How should a clinician manage a separated ins... 0.005    0.110   0.269       16.94              0.623             0.667 se